In [ ]:
from pyspark.sql import functions as F


tempo_zero = (
    (F.col("tipo_censura") == "exata")
    & (F.col("dur_min") == 0)
)

autotransicao = F.col("origem") == F.col("destino")

perfil_tempos = (
    transicoes_sm
    .groupBy("origem")
    .agg(
        F.count("*").alias("n_registros"),
        F.sum(
            F.when(tempo_zero, 1).otherwise(0)
        ).alias("n_zero"),
        F.sum(
            F.when(autotransicao, 1).otherwise(0)
        ).alias("n_autotransicoes"),
        F.sum(
            F.when(
                F.col("tipo_censura") == "direita", 1
            ).otherwise(0)
        ).alias("n_direita"),
    )
    .withColumn(
        "pct_zero",
        F.round(100 * F.col("n_zero") / F.col("n_registros"), 2),
    )
    .cache()  # Apenas o agregado de até 279 linhas.
)

perfil_tempos.orderBy(
    F.desc("n_zero")
).show(15, truncate=False)

In [ ]:
ESTADO_PILOTO = "sem_acao:::classe"

perfil_piloto = (
    perfil_tempos
    .filter(F.col("origem") == ESTADO_PILOTO)
    .first()
)

if perfil_piloto is None:
    raise ValueError(f"Estado não encontrado: {ESTADO_PILOTO}")

if perfil_piloto["n_zero"] > 0:
    raise ValueError(
        "O estado piloto possui durações exatas iguais a zero. "
        "Precisamos esclarecer sua origem antes do ajuste."
    )

if perfil_piloto["n_autotransicoes"] > 0:
    raise ValueError(
        "O estado piloto possui autotransições. "
        "Precisamos revisar a construção das permanências."
    )

# Seleção determinística de aproximadamente 0,1% dos clientes.
cliente_selecionado = (
    F.pmod(F.xxhash64("cd_bv"), F.lit(1000)) == 0
)

piloto_temporal = (
    transicoes_sm
    .filter(cliente_selecionado)
    .filter(F.col("origem") == ESTADO_PILOTO)
    .groupBy(
        "origem",
        "tipo_censura",
        "dur_min",
        "dur_max",
    )
    .agg(F.count("*").alias("peso"))
)

piloto_temporal.createOrReplaceGlobalTempView(
    "nba_sm_tempo_piloto"
)

In [ ]:
%r

library(survival)

max_grupos <- 200000L

consulta <- sprintf(
  "
  SELECT
      origem,
      tipo_censura,
      dur_min,
      dur_max,
      peso
  FROM global_temp.nba_sm_tempo_piloto
  LIMIT %d
  ",
  max_grupos + 1L
)

dados <- dplyr::collect(
  sparklyr::sdf_sql(sc, consulta)
)

if (nrow(dados) == 0L) {
  stop("A amostra do estado piloto ficou vazia.")
}

if (nrow(dados) > max_grupos) {
  stop("O piloto excedeu o limite de grupos. Reduza a amostra.")
}

if (any(
  dados$dur_min == 0 &
    dados$tipo_censura %in% c("exata", "direita")
)) {
  stop("Existem tempos zero que exigem tratamento antes do ajuste.")
}

# Conversão de segundos para dias.
dados$inferior <- dados$dur_min / 86400

dados$superior <- ifelse(
  dados$tipo_censura == "direita",
  Inf,
  dados$dur_max / 86400
)

# Para uma distribuição positiva, (0, U] equivale a T <= U.
intervalo_desde_zero <- (
  dados$tipo_censura == "intervalo" &
    dados$inferior == 0
)

dados$inferior[intervalo_desde_zero] <- NA_real_

# Interrompe em caso de aviso numérico, inclusive não convergência.
ajuste_tempo <- withCallingHandlers(
  survival::survreg(
    Surv(inferior, superior, type = "interval2") ~ 1,
    data = dados,
    weights = peso,
    dist = "weibull",
    na.action = na.fail,
    control = survival::survreg.control(maxiter = 100)
  ),
  warning = function(w) {
    stop(conditionMessage(w), call. = FALSE)
  }
)

if (!is.null(ajuste_tempo$fail)) {
  stop(ajuste_tempo$fail)
}

# Conversão para a parametrização de pweibull().
forma <- 1 / ajuste_tempo$scale
escala_dias <- exp(unname(coef(ajuste_tempo)[1]))

if (!all(is.finite(c(forma, escala_dias))) ||
    forma <= 0 || escala_dias <= 0) {
  stop("O ajuste retornou parâmetros inválidos.")
}

print(
  data.frame(
    estado = unique(dados$origem),
    registros_na_amostra = sum(dados$peso),
    grupos_de_duracao = nrow(dados),
    forma = forma,
    escala_dias = escala_dias
  )
)

In [ ]:
%r

tempo_decorrido <- 5
horizonte <- 7

log_s_atual <- pweibull(
  tempo_decorrido,
  shape = forma,
  scale = escala_dias,
  lower.tail = FALSE,
  log.p = TRUE
)

log_s_futuro <- pweibull(
  tempo_decorrido + horizonte,
  shape = forma,
  scale = escala_dias,
  lower.tail = FALSE,
  log.p = TRUE
)

probabilidade_saida <- -expm1(log_s_futuro - log_s_atual)

print(
  data.frame(
    dias_ja_em_silencio = tempo_decorrido,
    horizonte_dias = horizonte,
    probabilidade_de_sair_do_silencio = probabilidade_saida
  )
)

In [ ]:
%r

familias <- c(
  "weibull",
  "exponential",
  "lognormal",
  "loglogistic"
)

ajustar_familia <- function(familia) {
  ajuste <- withCallingHandlers(
    survival::survreg(
      survival::Surv(
        inferior,
        superior,
        type = "interval2"
      ) ~ 1,
      data = dados,
      weights = peso,
      dist = familia,
      na.action = na.fail,
      control = survival::survreg.control(maxiter = 100)
    ),
    warning = function(w) {
      stop(
        paste(familia, conditionMessage(w), sep = ": "),
        call. = FALSE
      )
    }
  )

  if (!is.null(ajuste$fail)) {
    stop(paste(familia, ajuste$fail, sep = ": "))
  }

  parametros <- c(coef(ajuste), ajuste$scale)

  if (any(!is.finite(parametros)) || ajuste$scale <= 0) {
    stop(paste("Parâmetros inválidos no ajuste:", familia))
  }

  if (!is.finite(as.numeric(logLik(ajuste)))) {
    stop(paste("Log-verossimilhança inválida:", familia))
  }

  ajuste
}

modelos_tempo <- setNames(
  lapply(familias, ajustar_familia),
  familias
)

comparacao <- do.call(
  rbind,
  lapply(names(modelos_tempo), function(familia) {
    ajuste <- modelos_tempo[[familia]]
    log_verossimilhanca <- logLik(ajuste)

    data.frame(
      distribuicao = familia,
      n_parametros = attr(log_verossimilhanca, "df"),
      log_verossimilhanca = as.numeric(log_verossimilhanca),
      AIC = AIC(ajuste)
    )
  })
)

comparacao$delta_AIC <- comparacao$AIC - min(comparacao$AIC)
comparacao <- comparacao[order(comparacao$AIC), ]
rownames(comparacao) <- NULL

print(comparacao)